# TP 2 — Profiling & périmètre

**Objectif** : connaître les données à fond, puis décider sur quels produits portera NutriScope.

1.	Profiling : distributions, cardinalités, doublons, unités, valeurs impossibles
2.	Inventaire des colonnes : utiles ou bruit ? (avec data-fields.txt d'Open Food Facts)
3.	Décider du périmètre : 5 à 8 rayons, colonnes gardées, seuil de complétude
4.	Rédiger le périmètre et ses justifications
5.	Présentation devant une autre équipe

In [1]:
# Commencer par faire les import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Les PATH pour les fichiers
RAW_PATH = "../../data/food.parquet"
FR_PATH = "../../data/food_fr.parquet"

# Les colonnes
AXES = [
    "code",
    "product_name",       # nouveau
    "brands",
    "categories_tags",
    "food_groups_tags",   # nouveau
    "countries_tags",
    "nutriscore_grade",
    "nutriments",
]


# Les différents nutriments gardés
NUTRIENTS = [
    "energy-kcal", 
    "fat", 
    "saturated-fat", 
    "carbohydrates", 
    "sugars", 
    "fiber", 
    "proteins", 
    "salt"
]

In [2]:
# Récupérer le DF brut
df_brut = pd.read_parquet(RAW_PATH, columns=AXES)

In [3]:
# Créer un masque pour les données en France
is_france = df_brut['countries_tags'].apply(lambda x : x is not None and 'en:france' in x)

In [4]:
# Créer le DF Français et retirer le DF brut de la RAM
df_fr = df_brut[is_france].copy()
del df_brut

In [5]:
# Récupérer la fonction du TP1
def get_100g(nutrients, name):
    """Renvoie la valeur pour 100 g du nutriment `name`, ou NaN s'il est absent."""
    if nutrients is None:
        return np.nan
    for nutrient in nutrients:
        if nutrient["name"] == name:
            return nutrient["100g"]
    return np.nan

In [6]:
# Appliqué la fonction à chaque ligne et créer une colonne attitrée (exploser)
for name in NUTRIENTS:
    df_fr[name + "_100g"] = df_fr["nutriments"].apply(lambda nutrients: get_100g(nutrients, name))

In [7]:
# On retire les colonnes nutriments et countries_tags car nutriments a été explosée et nous travaillons que sur le DF
df_fr = df_fr.drop(columns=["nutriments", "countries_tags"])
# Puis nous l'exportons en .parquet
df_fr.to_parquet(FR_PATH, index=False)

In [8]:
# On recharge le parquet Français
df = pd.read_parquet(FR_PATH)
df.head()

,code,product_name,brands,categories_tags,food_groups_tags,nutriscore_grade,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g
0,0000101209159,"[{'lang': 'main', 'text': 'Véritable pâte à ta...",Bovetti,"[en:breakfasts, en:spreads, en:sweet-spreads, ...","[en:sugary-snacks, en:sweets]",e,617.0,48.0,10.00,36.000000,32.0,NaN,8.00,0.01
1,0000130008136,"[{'lang': 'main', 'text': 'Escalope de dinde'}...",NaN,"[en:meats-and-their-products, en:meats, en:pou...","[en:fish-meat-eggs, en:meat, en:poultry]",unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0000140323687,"[{'lang': 'main', 'text': 'Madeleine Framboise...",NaN,"[en:snacks, en:sweet-snacks, en:biscuits-and-c...","[en:sugary-snacks, en:biscuits-and-cakes]",unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0000141013129,"[{'lang': 'main', 'text': 'Croissants margarin...",,"[en:snacks, en:sweet-snacks, en:sweet-pastries...","[en:sugary-snacks, en:pastries]",unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0000171812457,"[{'lang': 'main', 'text': 'Glaces vegetales de...",NaN,None,[],unknown,200.0,9.2,6.21,27.110001,26.6,NaN,2.22,0.10


In [9]:
df.shape # 1_247_336 lignes, 12 colonnes

(1247336, 14)

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1247336 entries, 0 to 1247335
Data columns (total 14 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   code                1247336 non-null  str    
 1   product_name        1247336 non-null  object 
 2   brands              704417 non-null   str    
 3   categories_tags     661380 non-null   object 
 4   food_groups_tags    1222308 non-null  object 
 5   nutriscore_grade    1222296 non-null  str    
 6   energy-kcal_100g    887859 non-null   float64
 7   fat_100g            880864 non-null   float64
 8   saturated-fat_100g  879680 non-null   float64
 9   carbohydrates_100g  881125 non-null   float64
 10  sugars_100g         880782 non-null   float64
 11  fiber_100g          379988 non-null   float64
 12  proteins_100g       882511 non-null   float64
 13  salt_100g           826467 non-null   float64
dtypes: float64(8), object(3), str(3)
memory usage: 161.5+ MB


In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
energy-kcal_100g,887859.0,1.603036e+10,1.510481e+13,-113.860001,114.0000,262.00,400.000000,1.423270e+16
fat_100g,880864.0,1.432543e+01,1.846786e+01,-0.001812,1.0000,8.10,22.500000,3.257000e+03
saturated-fat_100g,879680.0,5.420682e+00,8.151184e+00,-3.625000,0.2000,2.02,8.000000,1.238000e+03
carbohydrates_100g,881125.0,2.701358e+01,2.874133e+01,0.000000,2.6800,14.00,51.200001,4.010000e+03
sugars_100g,880782.0,1.333721e+27,1.251697e+30,0.000000,0.6000,3.50,19.780001,1.174717e+33
fiber_100g,379988.0,3.096741e+00,1.239176e+01,-0.424096,0.0135,1.50,3.600000,6.636000e+03
proteins_100g,882511.0,9.065288e+00,4.503649e+01,-0.005437,1.5000,6.30,13.000000,4.093900e+04
salt_100g,826467.0,1.278683e+00,9.205121e+00,0.000000,0.0750,0.55,1.300000,5.000000e+03


In [12]:
df['code'].duplicated(keep=False).sum() # 54 doublons des codes

np.int64(54)

In [13]:
df[df['code'].duplicated(keep=False)].sort_values('code')

,code,product_name,brands,categories_tags,food_groups_tags,nutriscore_grade,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g
4238,0059527070430,"[{'lang': 'main', 'text': 'Goji Snack Bars'}, ...",Taste Of Nature,"[en:snacks, en:sweet-snacks, en:bars, en:cerea...","[en:sugary-snacks, en:biscuits-and-cakes]",unknown,413.000000,17.299999,NaN,48.000000,NaN,NaN,16.700001,NaN
1058458,0059527070430,"[{'lang': 'main', 'text': 'Goji'}, {'lang': 'f...",Taste of Nature,"[en:snacks, en:sweet-snacks, en:bars, fr:barre...","[en:sugary-snacks, en:biscuits-and-cakes]",c,459.000000,28.000000,3.60,38.000000,27.00,7.600000,13.000000,0.10
4243,0059527171687,"[{'lang': 'main', 'text': 'Tast Of Nature Pome...",Taste of nature,[],[],unknown,434.000000,25.000000,3.30,49.000000,25.00,5.400000,11.000000,0.00
1058459,0059527171687,"[{'lang': 'main', 'text': 'Pomegranate'}, {'la...",Taste of Nature,"[en:snacks, en:sweet-snacks, en:bars, fr:barre...","[en:sugary-snacks, en:biscuits-and-cakes]",c,441.000000,25.000000,3.30,43.000000,27.00,6.100000,10.000000,0.04
4256,0059527401555,"[{'lang': 'main', 'text': 'Almond Fruit And Nu...",Taste Of Nature,"[en:snacks, en:sweet-snacks, en:bars, fr:snack...","[en:sugary-snacks, en:biscuits-and-cakes]",c,447.000000,25.000000,2.70,40.000000,26.00,8.200000,11.800000,0.04
1058453,0059527401555,"[{'lang': 'main', 'text': 'Almond'}, {'lang': ...",Taste of Nature,"[en:snacks, en:sweet-snacks, en:bars, fr:barre...","[en:sugary-snacks, en:biscuits-and-cakes]",c,447.000000,25.000000,2.70,40.000000,26.00,8.200000,12.000000,0.04
4257,0059527501552,"[{'lang': 'main', 'text': 'Cranberry Fruit and...",Taste Of Nature,"[en:plant-based-foods-and-beverages, en:plant-...","[en:sugary-snacks, en:biscuits-and-cakes]",c,462.000000,27.000000,3.60,38.000000,24.00,78.000000,13.600000,0.03
1058449,0059527501552,"[{'lang': 'main', 'text': 'Cranberry'}, {'lang...",Taste of Nature,"[en:snacks, en:sweet-snacks, en:bars, fr:barre...","[en:sugary-snacks, en:biscuits-and-cakes]",c,462.000000,27.000000,3.60,38.000000,24.00,7.800000,14.000000,0.03
4259,0059527601559,"[{'lang': 'main', 'text': 'Coconut, Cranberry,...",Taste Of Nature,"[en:plant-based-foods-and-beverages, en:plant-...","[en:sugary-snacks, en:biscuits-and-cakes]",d,NaN,27.000000,4.20,49.000000,28.00,5.700000,14.000000,0.00
1058456,0059527601559,"[{'lang': 'main', 'text': 'Brazil nut'}, {'lan...",Taste of Nature,"[en:snacks, en:sweet-snacks, en:bars, fr:barre...","[en:sugary-snacks, en:biscuits-and-cakes]",c,460.000000,28.000000,4.30,40.000000,26.00,6.000000,12.000000,0.03


In [14]:
COLS = ['sugars_100g', "salt_100g", "fat_100g"]

In [15]:
(df[COLS] > 100).sum()
# sugars_100g    47
# salt_100g      54
# fat_100g       44
# dtype: int64

sugars_100g    47
salt_100g      54
fat_100g       44
dtype: int64

In [16]:
df[COLS].loc[(df[COLS] > 100).any(axis=1)]

,sugars_100g,salt_100g,fat_100g
1493,0.000000,45.000000,533.333313
1649,35.714287,178.571426,25.000000
4739,170.000000,0.000042,0.000000
4782,300.000000,0.002175,500.000000
5605,0.000000,320.000000,0.000000
...,...,...,...
1210411,282.200012,0.560000,22.000000
1212701,21.000000,0.130000,123.800003
1217476,594.000000,0.090000,0.500000
1225764,38.000000,288.000000,0.000000


In [17]:
(df[COLS] < 0).sum()
# sugars_100g    0
# salt_100g      0
# fat_100g       1
# dtype: int64

sugars_100g    0
salt_100g      0
fat_100g       1
dtype: int64

In [18]:
df[COLS].loc[(df[COLS] < 0).any(axis=1)]

,sugars_100g,salt_100g,fat_100g
775002,25.25285,0.003751,-0.001812


In [19]:
(df['energy-kcal_100g'] > 900).sum() # 1116

np.int64(1116)

In [20]:
df.columns

Index(['code', 'product_name', 'brands', 'categories_tags', 'food_groups_tags',
       'nutriscore_grade', 'energy-kcal_100g', 'fat_100g',
       'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g',
       'proteins_100g', 'salt_100g'],
      dtype='str')

In [21]:
df[['energy-kcal_100g', 'categories_tags']][(df['energy-kcal_100g'] > 900)]

,energy-kcal_100g,categories_tags
160,8700.000000,"[en:snacks, en:sweet-snacks, en:biscuits-and-c..."
562,929.000000,"[en:plant-based-foods-and-beverages, en:plant-..."
563,928.571411,"[en:plant-based-foods-and-beverages, en:plant-..."
1493,8666.666992,[en:snacks]
1608,1920.000000,"[en:snacks, en:salty-snacks, en:appetizers, en..."
...,...,...
1244175,1000.000000,None
1244263,1191.000000,"[en:plant-based-foods-and-beverages, en:plant-..."
1244272,1233.000000,"[en:plant-based-foods-and-beverages, en:plant-..."
1245280,996.000000,"[en:plant-based-foods-and-beverages, en:plant-..."


In [22]:
df['categories_tags'].apply(lambda tags : tags is None or len(tags) == 0).sum() # 630082

np.int64(630082)

In [23]:
df.iloc[df['categories_tags'].apply(lambda tags : tags is None or len(tags) == 0)]

,code,product_name,brands,categories_tags,food_groups_tags,nutriscore_grade,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g
4,0000171812457,"[{'lang': 'main', 'text': 'Glaces vegetales de...",NaN,None,[],unknown,200.0,9.20,6.21,27.110001,26.6,NaN,2.220000,0.10
11,0000310413620,"[{'lang': 'main', 'text': 'Lomo de bellota ib...",NaN,None,[],unknown,316.0,15.09,6.50,0.540000,0.2,NaN,44.400002,2.71
14,0000604007221,"[{'lang': 'main', 'text': 'Pain Constance aux ...",NaN,None,[],unknown,134.0,3.00,0.30,47.900002,3.8,NaN,9.400000,1.30
15,0000605013450,"[{'lang': 'main', 'text': 'Le lin'}, {'lang': ...",NaN,None,[],unknown,294.0,5.40,0.60,48.000000,3.3,NaN,11.000000,1.50
18,0000653007548,"[{'lang': 'main', 'text': 'Baguettine aux oliv...",NaN,None,[],unknown,298.0,5.30,1.00,52.000000,2.0,NaN,8.300000,1.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1247331,3760340550999,"[{'lang': 'main', 'text': 'Biere Sans Alcool 7...",NaN,[],[],unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1247332,7707370942246,"[{'lang': 'main', 'text': 'Agua Potable Tratad...",NaN,None,[],unknown,0.0,0.00,0.00,0.000000,0.0,0.0,0.000000,0.00
1247333,8720877258428,"[{'lang': 'main', 'text': 'Spearmint'}, {'lang...",NaN,None,[],unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1247334,00276003,"[{'lang': 'main', 'text': 'Pure Whey Protein'}...",NaN,None,[],unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
categories = df['categories_tags'].explode()
categories.value_counts().head(30)

categories_tags
en:plant-based-foods-and-beverages         190606
en:plant-based-foods                       165363
en:snacks                                  108197
en:sweet-snacks                             90947
en:meats-and-their-products                 83549
en:beverages                                70703
en:meats                                    63989
en:dairies                                  59861
en:fruits-and-vegetables-based-foods        51085
en:cereals-and-potatoes                     50203
en:fermented-foods                          48850
en:fermented-milk-products                  47365
en:meals                                    45666
en:spreads                                  42476
en:biscuits-and-cakes                       39849
en:desserts                                 38759
en:prepared-meats                           37946
en:condiments                               37444
en:cheeses                                  33826
en:breakfasts                     

In [25]:
categories.iloc[categories.str.contains("snack")]

2                en:snacks
2          en:sweet-snacks
3                en:snacks
3          en:sweet-snacks
5                en:snacks
                ...       
1246686    en:sweet-snacks
1246749          en:snacks
1246749    en:salty-snacks
1247193          en:snacks
1247193    en:sweet-snacks
Name: categories_tags, Length: 219474, dtype: str

In [26]:
group_tag = df['food_groups_tags'].explode()

In [27]:
group_tag.head(100)

0      en:sugary-snacks
0             en:sweets
1     en:fish-meat-eggs
1               en:meat
1            en:poultry
            ...        
63    en:fish-meat-eggs
63    en:processed-meat
64                  NaN
65                  NaN
66                  NaN
Name: food_groups_tags, Length: 100, dtype: str

In [28]:
# Afficher les valeurs uniques qui ne contiennent pas 'en:' dans group_tag
group_tag.iloc[(~group_tag.str.contains('en:'))].unique()

<ArrowStringArray>
[nan]
Length: 1, dtype: str

In [29]:
def get_name(names):
    """Renvoie le nom en français, sinon le nom principal, sinon NaN."""
    if names is None:
        return np.nan
    main_name = np.nan
    for name in names:
        if name["lang"] == "fr":
            return name["text"]
        if name["lang"] == "main":
            main_name = name["text"]
    return main_name

In [30]:
def get_food_group(groups):
    """Renvoie le groupe alimentaire de niveau 1 (le rayon), ou NaN."""
    if groups is None or len(groups) == 0:
        return np.nan
    return groups[0]

In [31]:
# DF français et applique chacunes des fonctions
# Récupérer le nom
df_fr["product_name"] = df_fr["product_name"].apply(get_name)
# Récupérer le premier groupe alimentaire.
df_fr["food_group"] = df_fr["food_groups_tags"].apply(get_food_group)
# Maintenant que les colonnes product_name et food_groups_tags ont été traitées, nous pouvons retirer food_groups_tags, car product_name a été écrasé.
df_fr = df_fr.drop(columns=["food_groups_tags"])
# Appliquer les changements au fichier. (écrasement)
df_fr.to_parquet(FR_PATH, index=False) # Ce fichier nous servira de support pour le reste.

In [33]:
# Notre sélection des groupes
FOOD_GROUPS = [
    "en:sugary-snacks",
    "en:fish-meat-eggs",
    "en:cereals-and-potatoes",
    "en:milk-and-dairy-products",
    "en:beverages",
    "en:fats-and-sauces",
    "en:fruits-and-vegetables",
    "en:composite-foods",
    "en:salty-snacks",
]

# Récupérer les groupes qui sont dans notre liste
in_groups = df_fr["food_group"].isin(FOOD_GROUPS)
is_complete = (
    df_fr["code"].notna()
    & df_fr["product_name"].notna()
    & df_fr["energy-kcal_100g"].notna()
)

df_perimeter = df_fr[in_groups & is_complete].copy()
df_perimeter.shape


(456484, 14)

In [36]:
comparison = pd.DataFrame({
    "avant": df.isna().mean() * 100,
    "après": df_perimeter.isna().mean() * 100,
}).round(2)
comparison

,avant,après
brands,43.53,14.86
carbohydrates_100g,29.36,0.47
categories_tags,46.98,0.00
code,0.00,0.00
energy-kcal_100g,28.82,0.00
fat_100g,29.38,0.46
fiber_100g,69.54,47.41
food_group,NaN,0.00
food_groups_tags,2.01,NaN
nutriscore_grade,2.01,0.00


In [ ]:
round(df_perimeter['nutriscore_grade'].isin(list('abcde')).mean() * 100, 2)

np.float64(92.44)

In [47]:
df_perimeter['nutriscore_grade'].value_counts(normalize=True, dropna=False) * 100

nutriscore_grade
e                 26.547042
d                 24.580927
c                 19.162775
a                 12.300541
b                  9.851386
unknown            6.472078
not-applicable     1.085252
Name: proportion, dtype: float64

1. Que deviennent les manquants sur le sucre et le sel ? Pourquoi un filtre sur seulement 3 colonnes améliore-t-il autant toutes les autres ?

Prenons 'energy-kcal_100g' comme exemple, il y a des chances que si le code barre ne soit pas renseigné, que le reste des informations le soient également, ce qui forcément impact le nombre de valeur absente dans le reste des colonnes. Par conséquent, retirer les lignes où le code est absent, et donc l'article est inutilisable, nous baissons le nombre de valeurs absentes dans le reste des colonnes.

2. Quelle colonne reste mal remplie ? Le document de l'équipe parle d'« absence = souvent pas de fibres » : qu'en penses-tu ?

fiber_100g. Une valeur non renseignée n'est pas forcément mauvaise.

3. Quelle part du périmètre a un Nutri-Score de A à E ? Compare avec les 37 % du TP1.

92.44% sont entre A et E (a et e). Grosse différence entre 37.17% du TP1.